In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

In [65]:
df = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\archive\LOCMM_06062025_Baseline1.csv")

## Data processing

In [4]:
# Tách năm bắt đầu thành cột số nguyên để dễ xử lý
df['YearStart'] = df['YearID'].str[:4].astype(int)

# Xử lý cho dữ liệu từ 2001–2012
df1 = df[df['YearStart'] < 2013].copy()
df1.loc[df1['SmokeFam'] == 2, 'SmokeFam'] = 1
df1 = df1[df1['SmokeFam'] <= 1]

# Xử lý cho dữ liệu từ 2013 trở đi
df2 = df[df['YearStart'] >= 2013].copy()
df2.loc[(df2['SmokeFam'] >= 1.0) & (df2['SmokeFam'] <= 3.0), 'SmokeFam'] = 1
df2 = df2[df2['SmokeFam'] <= 1]
df = pd.concat([df1,df2])


In [5]:
#concat the valid 
df = pd.concat([df1,df2])

In [6]:
#diebete assign:
def isDiabete(row):
    if row<6.5:
        return 0
    else:
        return 1

In [7]:
df['Hba1c'] = df['Hba1c'].apply(isDiabete)

In [8]:
df.dropna(inplace=True)

In [9]:
def labelVitaminD(row):
    if row<50:
        return 1
    else:
        return 0
    

In [10]:
df['label'] = df['VitaminD'].apply(labelVitaminD)

In [11]:
df['label'].value_counts()

label
0    27525
1    15881
Name: count, dtype: int64

In [12]:
df.drop(columns=['SEQN','VitaminD','YearStart','YearID'],inplace=True)

In [13]:
df = df[df['milk_consumption']<=3]

In [14]:
df.columns

Index(['Gender', 'Age', 'Race', 'familysize', 'PIR', 'BMI', 'Hba1c',
       'milk_consumption', 'SmokeFam', 'label'],
      dtype='object')

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Assuming `df` is your original dataframe
# X = df.drop(columns=["label"])
# y = df["label"]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.3,
#     stratify=y,
#     random_state=42
# )

# # Combine features and label for export
# train_df = pd.concat([X_train, y_train], axis=1)
# val_df = pd.concat([X_test, y_test], axis=1)

# df = pd.concat([train_df,val_df])

# === Bước 3: Tách dữ liệu theo mốc năm 2013 ===
df_final_train = df[df['YearStart'] <= 2013]
df_final_test = df[df['YearStart'] > 2013]

# === Bước 4: Ghi ra file CSV ===
# df_final_train.to_csv(r'c:\mydata\G8Vitamin\data\final\baselinestore\processed_train.csv', index=False)
# df_final_test.to_csv(r'c:\mydata\G8Vitamin\data\final\baselinestore\processed_test.csv', index=False)

# === Log số dòng để xác nhận ===
print(f"✅ Số dòng train: {len(df_final_train)} được lưu vào train.csv")
print(f"✅ Số dòng test : {len(df_final_test)} được lưu vào test.csv")


KeyError: 'YearStart'

In [98]:
df_train = pd.read_csv(r'c:\mydata\G8Vitamin\data\final\baselinestore\processed_train_v2.csv')
df_test=pd.read_csv(r'c:\mydata\G8Vitamin\data\final\baselinestore\processed_test_v2.csv')

# Models

In [100]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss
)

# === 1. Load Data ===
# df = pd.read_csv("your_file.csv")  # <-- Replace with your actual filename

# === 2. Features and Label ===
X = df.drop(columns=["label"])
y = df["label"].values


# === 3. Define categorical and numerical features ===
categorical_features = ["Gender", "Race", "milk_consumption", "Hba1c", "SmokeFam"]
numerical_features = [col for col in X.columns if col not in categorical_features]

# === 4. Preprocessing ===
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

# === 5. Models ===
models = {
    "GBM": GradientBoostingClassifier(),
    "LR": LogisticRegression(max_iter=1000),
    "Nnet": MLPClassifier(max_iter=1000),
    "RF": RandomForestClassifier(),
    # "SVM": SVC(probability=True),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss")
}

# === 6. Evaluation Function ===
def evaluate_model(name, model, X, y):
    sss = StratifiedShuffleSplit(n_splits=10, test_size=0.3, random_state=42)
    metrics = {
        "AUC": [], "ACC": [], "PPV": [], "NPV": [], "SEN": [],
        "SPE": [], "F1 score": [], "MCC": [], "KAPPA": [], "Brier score": []
    }

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    for train_idx, test_idx in sss.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        metrics["AUC"].append(roc_auc_score(y_test, y_proba))
        metrics["ACC"].append(accuracy_score(y_test, y_pred))
        metrics["PPV"].append(precision_score(y_test, y_pred, zero_division=0))
        metrics["NPV"].append(tn / (tn + fn) if (tn + fn) > 0 else 0)
        metrics["SEN"].append(recall_score(y_test, y_pred))  # Sensitivity
        metrics["SPE"].append(tn / (tn + fp) if (tn + fp) > 0 else 0)  # Specificity
        metrics["F1 score"].append(f1_score(y_test, y_pred))
        metrics["MCC"].append(matthews_corrcoef(y_test, y_pred))
        metrics["KAPPA"].append(cohen_kappa_score(y_test, y_pred))
        metrics["Brier score"].append(brier_score_loss(y_test, y_proba))

    return {metric: np.mean(values) for metric, values in metrics.items()}

# === 7. Run All Models ===
results = {}
for name, model in models.items():
    print(f"Evaluating {name}...")
    results[name] = evaluate_model(name, model, X, y)

# === 8. Export Results ===
results_df = pd.DataFrame(results).T
results_df.index.name = "Method"
results_df.to_csv("evaluation_results.csv")
print(results_df)


KeyError: "['label'] not found in axis"

# Test on split data

In [107]:
df_final_train = pd.read_csv(r'c:\mydata\G8Vitamin\data\final\baselinestore\processed_train.csv')
df_final_test  = pd.read_csv(r'c:\mydata\G8Vitamin\data\final\baselinestore\processed_test.csv')


In [108]:
threshold = 1e-10

# Select numeric columns
numeric_cols = df_final_train.select_dtypes(include='number').columns

# Apply threshold replacement only on numeric columns
df_final_train[numeric_cols] = df_final_train[numeric_cols].applymap(
    lambda x: 0 if abs(x) < threshold else x
)
df_final_test[numeric_cols] = df_final_test[numeric_cols].applymap(
    lambda x: 0 if abs(x) < threshold else x
)


C:\Users\iseT1enLoc\AppData\Local\Temp\ipykernel_13716\2484542337.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_final_train[numeric_cols] = df_final_train[numeric_cols].applymap(
C:\Users\iseT1enLoc\AppData\Local\Temp\ipykernel_13716\2484542337.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_final_test[numeric_cols] = df_final_test[numeric_cols].applymap(


In [109]:
df_final_train['SmokeFam'] = df_final_train.apply(
    lambda row: 0 if row['SmokeFam'] == 2 and int(row['YearStart']) <= 2011 else 1,
    axis=1
)

In [110]:
df_final_test['SmokeFam'] = df_final_test.apply(
    lambda row: 0 if row['SmokeFam'] == 0 else 1,
    axis=1
)

In [111]:
df_final_train['SmokeFam'].value_counts()

SmokeFam
0    28669
1     7895
Name: count, dtype: int64

In [112]:
df_final_test['SmokeFam'].value_counts()

SmokeFam
0    4559
1    2213
Name: count, dtype: int64

In [113]:
columns_remove = [
    'VitaminD',
    'SEQN',
    'YearID',
    'YearStart',
]
df_final_train.drop(columns=columns_remove,inplace=True)
df_final_test.drop(columns=columns_remove,inplace=True)

In [114]:
df_final_train = df_final_train[df_final_train['milk_consumption']<=3]
df_final_test = df_final_test[df_final_test['milk_consumption']<=3]

In [115]:
df_final_train['SmokeFam'].value_counts()

SmokeFam
0    28555
1     7853
Name: count, dtype: int64

In [116]:
df_final_test['SmokeFam'].value_counts()

SmokeFam
0    4542
1    2207
Name: count, dtype: int64

In [ ]:
# df_final_train.to_csv(r'c:\mydata\G8Vitamin\data\final\baselinestore\processed_train_v2.csv',index=False)
# df_final_test.to_csv(r'c:\mydata\G8Vitamin\data\final\baselinestore\processed_test_v2.csv',index=False)

In [3]:
import pandas as pd
df_train = pd.read_csv(r'C:\mydata\G8Vitamin\data\final\baselinestore\processed_train_v2.csv')
df_test = pd.read_csv(r'C:\mydata\G8Vitamin\data\final\baselinestore\processed_test_v2.csv')

In [4]:
df_train['label'].value_counts()

label
0.0    27290
1.0     9118
Name: count, dtype: int64

In [5]:
df_test['label'].value_counts()

label
0.0    4581
1.0    2168
Name: count, dtype: int64

In [118]:
df_final_train.columns

Index(['Gender', 'Age', 'Race', 'familysize', 'PIR', 'BMI', 'Hba1c',
       'milk_consumption', 'SmokeFam', 'label'],
      dtype='object')

In [119]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss
)

# === 1. Features and Label ===
X_train = df_final_train.drop(columns=["label"])
y_train = df_final_train["label"].values
X_test  = df_final_test.drop(columns=["label"])
y_test  = df_final_test["label"].values

# === 2. Define categorical and numerical features ===
categorical_features = ["Gender", "Race", "milk_consumption", "SmokeFam"]
numerical_features = [col for col in X_train.columns if col not in categorical_features]

# === 3. Preprocessing ===
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

# === 4. Models ===
models = {
    "GBM": GradientBoostingClassifier(),
    "LR": LogisticRegression(max_iter=1000),
    "Nnet": MLPClassifier(max_iter=1000),
    "RF": RandomForestClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss")
}

# === 5. Evaluation Function (no random split) ===
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    results = {
        "AUC": roc_auc_score(y_test, y_proba),
        "ACC": accuracy_score(y_test, y_pred),
        "PPV": precision_score(y_test, y_pred, zero_division=0),
        "NPV": tn / (tn + fn) if (tn + fn) > 0 else 0,
        "SEN": recall_score(y_test, y_pred),  # Sensitivity (Recall for 1)
        "SPE": tn / (tn + fp) if (tn + fp) > 0 else 0,  # Specificity (Recall for 0)
        "F1_binary": f1_score(y_test, y_pred, average="binary"),
        "F1_macro": f1_score(y_test, y_pred, average="macro"),
        "F1_weighted": f1_score(y_test, y_pred, average="weighted"),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "KAPPA": cohen_kappa_score(y_test, y_pred),
        "Brier score": brier_score_loss(y_test, y_proba)
    }

    return results

# === 6. Run All Models ===
results = {}
for name, model in models.items():
    print(f"Evaluating {name}...")
    results[name] = evaluate_model(name, model, X_train, y_train, X_test, y_test)

# === 7. Export Results ===
results_df = pd.DataFrame(results).T
results_df.index.name = "Method"
results_df.to_csv("evaluation_results.csv")
print(results_df)


Evaluating GBM...
Evaluating LR...
Evaluating Nnet...
Evaluating RF...
Evaluating XGBoost...


c:\mydata\G8Vitamin\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [18:26:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              AUC       ACC       PPV       NPV       SEN       SPE  \
Method                                                                
GBM      0.737839  0.714328  0.636986  0.725864  0.257380  0.930583   
LR       0.727374  0.711809  0.618491  0.726928  0.268450  0.921633   
Nnet     0.716108  0.705882  0.589971  0.726448  0.276753  0.908972   
RF       0.711742  0.706771  0.589404  0.728566  0.287362  0.905261   
XGBoost  0.714873  0.714032  0.598673  0.739130  0.333026  0.894346   

         F1_binary  F1_macro  F1_weighted       MCC     KAPPA  Brier score  
Method                                                                      
GBM       0.366623  0.591098     0.671355  0.261156  0.222957     0.185832  
LR        0.374397  0.593590     0.671959  0.256239  0.223379     0.188580  
Nnet      0.376766  0.592145     0.669151  0.242418  0.215915     0.191750  
RF        0.386357  0.596858     0.672119  0.247484  0.222676     0.192808  
XGBoost   0.427979  0.618671     0.68685

# SMOTE

In [120]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss
)
from imblearn.over_sampling import SMOTE

# === 1. Features and Label ===
X_train = df_final_train.drop(columns=["label"])
y_train = df_final_train["label"].values
X_test  = df_final_test.drop(columns=["label"])
y_test  = df_final_test["label"].values

# === 2. Define categorical and numerical features ===
categorical_features = ["Gender", "Race", "milk_consumption", "SmokeFam"]
numerical_features = [col for col in X_train.columns if col not in categorical_features]

# === 3. Preprocessing ===
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

# === 4. Models ===
models = {
    "GBM": GradientBoostingClassifier(),
    "LR": LogisticRegression(max_iter=1000),
    "Nnet": MLPClassifier(max_iter=1000),
    "RF": RandomForestClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss")
}

# === 5. Evaluation Function (with SMOTE on training data only) ===
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    # 1. Apply preprocessing first
    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)

    # 2. Apply SMOTE on processed training data
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_processed, y_train)

    # 3. Train model
    model.fit(X_train_res, y_train_res)

    # 4. Predict
    y_pred = model.predict(X_test_processed)
    y_proba = model.predict_proba(X_test_processed)[:, 1]

    # 5. Metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    results = {
        "AUC": roc_auc_score(y_test, y_proba),
        "ACC": accuracy_score(y_test, y_pred),
        "PPV": precision_score(y_test, y_pred, zero_division=0),
        "NPV": tn / (tn + fn) if (tn + fn) > 0 else 0,
        "SEN": recall_score(y_test, y_pred),  # Sensitivity (Recall for 1)
        "SPE": tn / (tn + fp) if (tn + fp) > 0 else 0,  # Specificity
        "F1_binary": f1_score(y_test, y_pred, average="binary"),
        "F1_macro": f1_score(y_test, y_pred, average="macro"),
        "F1_weighted": f1_score(y_test, y_pred, average="weighted"),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "KAPPA": cohen_kappa_score(y_test, y_pred),
        "Brier score": brier_score_loss(y_test, y_proba)
    }

    return results

# === 6. Run All Models ===
results = {}
for name, model in models.items():
    print(f"Evaluating {name} with SMOTE...")
    results[name] = evaluate_model(name, model, X_train, y_train, X_test, y_test)

# === 7. Export Results ===
results_df = pd.DataFrame(results).T
results_df.index.name = "Method"
results_df.to_csv("evaluation_results_smote.csv")
print(results_df)


Evaluating GBM with SMOTE...
Evaluating LR with SMOTE...
Evaluating Nnet with SMOTE...
Evaluating RF with SMOTE...
Evaluating XGBoost with SMOTE...
              AUC       ACC       PPV       NPV       SEN       SPE  \
Method                                                                
GBM      0.720394  0.687361  0.511554  0.791460  0.592251  0.732373   
LR       0.727060  0.622463  0.449602  0.841222  0.781827  0.547042   
Nnet     0.687292  0.616536  0.440239  0.808037  0.713561  0.570618   
RF       0.707570  0.693732  0.526621  0.759068  0.460793  0.803973   
XGBoost  0.710682  0.705734  0.555966  0.753270  0.416974  0.842392   

         F1_binary  F1_macro  F1_weighted       MCC     KAPPA  Brier score  
Method                                                                      
GBM       0.548953  0.654862     0.692728  0.313633  0.311675     0.201004  
LR        0.570899  0.616931     0.633389  0.309262  0.275293     0.230151  
Nnet      0.544527  0.606705     0.628937  0.2

c:\mydata\G8Vitamin\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [18:28:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [34]:
df_test = pd.read_csv(r'C:\mydata\G8Vitamin\data\final\baselinestore\test.csv')
df_train=pd.read_csv(r'C:\mydata\G8Vitamin\data\final\baselinestore\train.csv')

In [35]:
df_test.describe()

,SEQN,Gender,Age,Race,familysize,PIR,BMI,Hba1c,milk_consumption,SmokeFam,VitaminD,YearStart,label
count,13145.000000,13145.000000,13145.000000,13145.000000,13145.000000,1.163100e+04,12954.000000,12371.000000,1.314500e+04,8.003000e+03,12297.000000,13145.000000,12297.000000
mean,93378.577938,1.514036,44.000533,3.142183,3.460936,2.415448e+00,28.734970,5.757699,1.784861e+00,1.664001e+00,65.163937,2015.973906,0.324632
std,5541.653028,0.499822,20.996784,1.293014,1.726512,1.593628e+00,7.381482,1.072404,1.099355e+00,3.386262e+01,29.302997,0.999698,0.468256
min,83732.000000,1.000000,12.000000,1.000000,1.000000,5.397605e-79,13.200000,3.800000,5.397605e-79,5.397605e-79,7.040000,2015.000000,0.000000
25%,88604.000000,1.000000,25.000000,2.000000,2.000000,1.090000e+00,23.500000,5.200000,1.000000e+00,5.397605e-79,44.800000,2015.000000,0.000000
50%,93443.000000,2.000000,44.000000,3.000000,3.000000,1.990000e+00,27.700000,5.500000,2.000000e+00,5.397605e-79,61.200000,2015.000000,0.000000
75%,98147.000000,2.000000,62.000000,4.000000,5.000000,3.730000e+00,32.600000,5.900000,3.000000e+00,1.000000e+00,79.900000,2017.000000,1.000000
max,102956.000000,2.000000,80.000000,5.000000,7.000000,5.000000e+00,86.200000,17.000000,9.000000e+00,9.990000e+02,422.000000,2017.000000,1.000000


In [94]:
df_final_train = pd.read_csv(r'c:\mydata\G8Vitamin\data\final\baselinestore\processed_train_v2.csv')
df_final_test  = pd.read_csv(r'c:\mydata\G8Vitamin\data\final\baselinestore\processed_test_v2.csv')


In [95]:
df_final_train['Hba1c'] = df_final_train['Hba1c'].apply(isDiabete)
df_final_test['Hba1c'] = df_final_test['Hba1c'].apply(isDiabete)

In [96]:
#diebete assign:
def isDiabete(row):
    if row<6.5:
        return 0
    else:
        return 1

In [97]:
import pandas as pd

# === Define categorical and numerical columns ===
categorical_cols = ["Gender", "Race", "familysize", "SmokeFam", "milk_consumption", "Hba1c"]
numerical_cols = ["Age", "PIR", "BMI"]  # adjust based on your dataframe

def summarize_dataframe(df, categorical_cols, numerical_cols):
    summary = []

    # Categorical columns: counts and percentages
    for col in categorical_cols:
        counts = df[col].value_counts(dropna=False)
        percents = df[col].value_counts(normalize=True, dropna=False) * 100
        for cat in counts.index:
            summary.append({
                "Variable": col,
                "Category": cat,
                "Count": counts[cat],
                "Percentage": f"{percents[cat]:.2f}%"
            })

    # Numerical columns: median (Q1,Q3)
    for col in numerical_cols:
        median = df[col].median()
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        summary.append({
            "Variable": col,
            "Category": "Median(Q1,Q3)",
            "Count": f"{median:.2f}({q1:.2f},{q3:.2f})",
            "Percentage": ""
        })

    summary_df = pd.DataFrame(summary)
    return summary_df

# === Generate summary tables ===
train_summary = summarize_dataframe(df_final_train, categorical_cols, numerical_cols)
test_summary  = summarize_dataframe(df_final_test, categorical_cols, numerical_cols)

# === Export to CSV if needed ===
train_summary.to_csv("train_summary.csv", index=False)
test_summary.to_csv("test_summary.csv", index=False)

# === Print head to check ===
print("Train Summary:")
print(train_summary.head(20))
print("\nTest Summary:")
print(test_summary.head(20))


Train Summary:
            Variable Category  Count Percentage
0             Gender      2.0  18498     50.81%
1             Gender      1.0  17910     49.19%
2               Race      3.0  16117     44.27%
3               Race      4.0   8289     22.77%
4               Race      1.0   7237     19.88%
5               Race      2.0   2464      6.77%
6               Race      5.0   2301      6.32%
7         familysize      2.0   9342     25.66%
8         familysize      4.0   6751     18.54%
9         familysize      3.0   6551     17.99%
10        familysize      5.0   4793     13.16%
11        familysize      1.0   3915     10.75%
12        familysize      7.0   2644      7.26%
13        familysize      6.0   2412      6.62%
14          SmokeFam        0  28555     78.43%
15          SmokeFam        1   7853     21.57%
16  milk_consumption      3.0  17142     47.08%
17  milk_consumption      2.0   9493     26.07%
18  milk_consumption      1.0   4917     13.51%
19  milk_consumption     